🔷 Question 2: Design a Multi-Agent Workflow with LangGraph (25 Marks)
🧩 Scenario
You are building an AI-powered customer support system for a fintech company.
The system must handle:
Transaction queries
Fraud detection flags
Refund requests
General FAQs
The company wants:
High accuracy
Step-by-step reasoning
Ability to retry if answer is incorrect
Modular, scalable architecture
💻 Task
Design and implement a multi-agent workflow using LangGraph (or similar framework).
✅ 1. Agent Design
Define at least 3 agents, such as:
Retrieval Agent
Reasoning/Answer Agent
Validation Agent
Explain briefly (in comments or code):
Each agent’s role
Input/output
✅ 2. Graph Workflow Implementation
Write code or pseudo-code to:
Define state
Add nodes (agents)
Define edges
Implement conditional logic
👉 Must include:
Retry loop if validation fails
Clear start and end states
✅ 3. State Management
Show how state evolves across steps:
Query
Context
Intermediate reasoning
Final answer
Validation flag
✅ 4. Task Delegation & Communication
Demonstrate:
How agents pass information
How decisions are made between agents
🎯 Expected Outcome
A clear multi-step, graph-based agent system that:
Handles complex queries
Demonstrates reasoning + validation
Uses proper orchestration

In [15]:
from langgraph.graph import StateGraph, END
from typing import TypedDict


# 1. Define State

class AgentState(TypedDict):
    query: str
    context: str
    reasoning: str
    answer: str
    is_valid: bool
    retry_count: int


# 2. Dummy LLM (simulation)

class DummyLLM:
    def invoke(self, prompt: str):
        return f"Generated response for: {prompt}"

llm = DummyLLM()



# 3. Agents


#  Retrieval Agent
def retrieval_agent(state: AgentState):
    query = state["query"]
    context = f"[DB] Financial data for: {query}"
    return {**state, "context": context}


# 🔹 Reasoning Agent
def reasoning_agent(state: AgentState):
    query = state["query"]
    context = state["context"]

    reasoning = f"Analyzing: {query}"
    answer = llm.invoke(f"{query} | Context: {context}")

    return {
        **state,
        "reasoning": reasoning,
        "answer": answer
    }


#  Validation Agent
def validation_agent(state: AgentState):
    answer = state["answer"]

    # Example validation rule
    is_valid = "error" not in answer.lower()

    return {**state, "is_valid": is_valid}


#  Retry Agent
def retry_agent(state: AgentState):
    return {
        **state,
        "retry_count": state["retry_count"] + 1
    }



# 4. Router Logic

def validation_router(state: AgentState):
    if state["is_valid"]:
        return "end"
    elif state["retry_count"] >= 2:
        return "end"
    else:
        return "retry"



# 5. Build Graph

graph = StateGraph(AgentState)

graph.add_node("retrieval", retrieval_agent)
graph.add_node("reasoning", reasoning_agent)
graph.add_node("validation", validation_agent)
graph.add_node("retry", retry_agent)

# Entry point
graph.set_entry_point("retrieval")

# Flow
graph.add_edge("retrieval", "reasoning")
graph.add_edge("reasoning", "validation")

# Conditional flow (FIXED)
graph.add_conditional_edges(
    "validation",
    validation_router,
    {
        "retry": "retry",
        "end": END   #
    }
)

# Retry loop
graph.add_edge("retry", "reasoning")

# Compile
app = graph.compile()


# -----------------------------
# 6. Run
# -----------------------------
if __name__ == "__main__":
    initial_state = {
        "query": "Why was my transaction declined?",
        "context": "",
        "reasoning": "",
        "answer": "",
        "is_valid": False,
        "retry_count": 0
    }

    result = app.invoke(initial_state)

    print("\nFINAL OUTPUT:\n")
    print(result)


FINAL OUTPUT:

{'query': 'Why was my transaction declined?', 'context': '[DB] Financial data for: Why was my transaction declined?', 'reasoning': 'Analyzing: Why was my transaction declined?', 'answer': 'Generated response for: Why was my transaction declined? | Context: [DB] Financial data for: Why was my transaction declined?', 'is_valid': True, 'retry_count': 0}
